In [1]:
import pandas as pd
import numpy as np
import math
from typing import List, Dict, Tuple

# Wilaya coordinates in Algeria
town_coords = {
    'Adrar': (27.8749, -0.2930),          'Chlef': (36.1650, 1.3347),
    'Laghouat': (33.8078, 2.8808),        'Oum El Bouaghi': (35.8752, 7.1136),
    'Batna': (35.5559, 6.1741),           'Béjaïa': (36.7509, 5.0567),
    'Biskra': (34.8603, 5.7288),          'Béchar': (31.6237, -2.2164),
    'Blida': (36.4706, 2.8287),           'Bouira': (36.3763, 3.9000),
    'Tamanrasset': (22.7903, 5.5193),     'Tébessa': (35.4072, 8.1200),
    'Tlemcen': (34.8828, -1.3167),        'Tiaret': (35.3700, 1.3200),
    'Tizi Ouzou': (36.7167, 4.0500),      'Algiers': (36.7372, 3.0872),
    'Djelfa': (34.6667, 3.2500),          'Jijel': (36.8206, 5.7667),
    'Sétif': (36.1914, 5.4136),           'Saïda': (34.8414, 0.1514),
    'Skikda': (36.8792, 6.9067),          'Sidi Bel Abbès': (35.1939, -0.6414),
    'Annaba': (36.9000, 7.7667),          'Guelma': (36.4667, 7.4333),
    'Constantine': (36.3650, 6.6147),     'Médéa': (36.2675, 2.7500),
    'Mostaganem': (35.9333, 0.0833),      'M\'Sila': (35.7058, 4.5419),
    'Mascara': (35.3983, 0.1467),         'Ouargla': (31.9500, 5.3167),
    'Oran': (35.6911, -0.6417),           'El Bayadh': (33.6833, 1.0167),
    'Illizi': (26.4833, 8.4667),          'Bordj Bou Arréridj': (36.0667, 4.7667),
    'Boumerdès': (36.7667, 3.4667),       'El Tarf': (36.7667, 8.3167),
    'Tindouf': (27.6742, -8.1478),        'Tissemsilt': (35.6072, 1.8106),
    'El Oued': (33.3683, 6.8672),         'Khenchela': (35.4167, 7.1333),
    'Souk Ahras': (36.2833, 7.9500),      'Tipaza': (36.5897, 2.4475),
    'Mila': (36.4503, 6.2644),            'Aïn Defla': (36.2642, 1.9678),
    'Naama': (33.2667, -0.3167),          'Aïn Témouchent': (35.3028, -1.1414),
    'Ghardaïa': (32.4833, 3.6667),        'Relizane': (35.7372, 0.5558),
    'El M\'Ghair': (33.9500, 5.9167),     'El Menia': (30.5833, 2.8833),
    'Ouled Djellal': (34.4333, 5.0667),   'Bordj Badji Mokhtar': (21.3167, 0.9500),
    'Béni Abbès': (30.1333, -2.1667),     'Timimoun': (29.2500, 0.2333),
    'Touggourt': (33.1000, 6.0667),       'Djanet': (24.5553, 9.4892),
    'In Salah': (27.2167, 2.4667),        'In Guezzam': (19.8500, 5.7333)
}



# Haversine formula to compute distance in kilometers
def haversine(coord1: Tuple[float, float], coord2: Tuple[float, float]) -> float:
    lat1, lon1 = np.radians(coord1)
    lat2, lon2 = np.radians(coord2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c

def load_data() -> pd.DataFrame:
    try:
        seekers_df = pd.read_csv('emplo.csv')
        if 'employee_Id' not in seekers_df.columns:
            seekers_df['employee_Id'] = [f'EMP-{i+1:03}' for i in range(len(seekers_df))]
    except FileNotFoundError:
        seekers_df = pd.DataFrame([{
            'employee_Id': 'EMP-001',
            'gender': 'Male', 'birth_date': '1990-05-15', 'age': 33,
            'city': 'Algiers', 'wilaya': 'Algiers',
            'department': 'IT', 'sector': 'Technology',
            'years_experience': 5, 'highest_education': 'Master',
            'contract_type': 'Full-time', 'technical_skills': 'Python, SQL, Spark',
            'language_proficiency': 'English, French',
            'education_history': 'University of Algiers',
            'edu_value': 3, 'salary': 70000
        }])
    
    # Add missing wilaya column
    if 'wilaya' not in seekers_df.columns:
        if 'city' in seekers_df.columns:
            seekers_df['wilaya'] = seekers_df['city']
        else:
            seekers_df['wilaya'] = 'Algiers'  # Default value
    
    seekers_df['wilaya'] = seekers_df['wilaya'].str.title()
    invalid_wilayas = seekers_df[~seekers_df['wilaya'].isin(town_coords)]
    if not invalid_wilayas.empty:
        print(f"\nWarning: Invalid wilayas in employee data: {invalid_wilayas['wilaya'].unique()}")
    
    return seekers_df
def input_job_details(seekers_df: pd.DataFrame) -> pd.DataFrame:
    print("\n=== Enter Single Job Details ===")
    print("Press Enter for defaults or provide custom values\n")
    
    job_id = input("Job ID [JOB-001]: ").strip() or "JOB-001"
    title = input("Job Title [Data Engineer]: ").strip() or "Data Engineer"
    req_skills = input("Required Skills (comma-separated) [Python, SQL, Spark]: ").strip()
    required_skills = [s.strip() for s in req_skills.split(',')] if req_skills else ['Python', 'SQL', 'Spark']
    
    min_exp = input("Minimum Experience Years [3]: ").strip()
    min_experience = int(min_exp) if min_exp.isdigit() else 3
    
    min_edu = input("Minimum Education [Master]: ").strip() or "Master"
    salary = input("Salary Offer [75000]: ").strip()
    salary_offer = float(salary) if salary.replace('.','',1).isdigit() else 75000.0
    
    location = ""
    while not location or location not in town_coords:
        location = input("Location/Wilaya [Algiers]: ").strip().title() or "Algiers"
        if location not in town_coords:
            print(f"Invalid wilaya. Try: Algiers, Oran, Constantine, etc.")
    
    sector = input("Sector [Technology]: ").strip() or "Technology"
    contract = input("Contract Type [Full-time]: ").strip() or "Full-time"

    return pd.DataFrame([{
        'job_id': job_id,
        'title': title,
        'required_skills': required_skills,
        'min_experience': min_experience,
        'min_education': min_edu,
        'salary_offer': salary_offer,
        'location': location,
        'sector': sector,
        'contract_type': contract
    }])



# ---------- 3. Core Matching Functions ----------
def preprocess_data(seekers_df: pd.DataFrame, jobs_df: pd.DataFrame):
    # education mapping
    mapping = seekers_df.groupby('highest_education')['edu_value'].first().to_dict()
    seekers_df['edu_rank'] = seekers_df['edu_value']
    jobs_df['edu_rank'] = jobs_df['min_education'].map(mapping).fillna(-1).astype(int)
    # skills
    seekers_df['technical_skills'] = seekers_df['technical_skills'].fillna('').str.lower()
    seeker_skills = [set(s.split(', ')) for s in seekers_df['technical_skills']]
    job_skills = [set(map(str.lower, req)) for req in jobs_df['required_skills']]
    # sector & contract codes
    all_sectors = pd.Categorical(seekers_df['sector'].tolist() + jobs_df['sector'].tolist())
    seek_sector_codes = pd.Categorical(seekers_df['sector'], categories=all_sectors.categories).codes
    job_sector_codes = pd.Categorical(jobs_df['sector'], categories=all_sectors.categories).codes
    all_contracts = pd.Categorical(seekers_df['contract_type'].tolist() + jobs_df['contract_type'].tolist())
    seek_cont_codes = pd.Categorical(seekers_df['contract_type'], categories=all_contracts.categories).codes
    job_cont_codes = pd.Categorical(jobs_df['contract_type'], categories=all_contracts.categories).codes
    # coordinates
    seekers_coords = seekers_df['wilaya'].map(lambda w: town_coords.get(w, (np.nan, np.nan))).tolist()
    jobs_coords = jobs_df['location'].map(lambda w: town_coords.get(w, (np.nan, np.nan))).tolist()
    return seekers_df, jobs_df, seeker_skills, job_skills, \
           seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes, \
           seekers_coords, jobs_coords

def calculate_features(seekers_df, jobs_df, seeker_skills, job_skills,
                       seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes,
                       seekers_coords, jobs_coords) -> np.ndarray:
    N_seek, N_job = len(seekers_df), len(jobs_df)
    F_raw = np.zeros((N_seek, N_job, 8), dtype=np.float32)
    # compute idf
    all_skills = [skill for skills in seekers_df['technical_skills'] for skill in skills.split(', ') if skill]
    counts = pd.Series(all_skills).value_counts()
    tot_docs = N_seek + N_job
    skill_idf = {s: np.log((tot_docs+1)/(cnt+1))+1 for s, cnt in counts.items()}
    # salary norm
    all_sals = np.concatenate([seekers_df['salary'], jobs_df['salary_offer']])
    max_sal, min_sal = np.percentile(all_sals, 95), np.percentile(all_sals,5)
    # edu norm
    max_edu = seekers_df['edu_value'].max()
    # distance matrix
    dist = np.full((N_seek, N_job), np.nan, dtype=np.float32)
    for i in range(N_seek):
        for j in range(N_job):
            c1, c2 = seekers_coords[i], jobs_coords[j]
            if not math.isnan(c1[0]) and not math.isnan(c2[0]):
                dist[i,j] = haversine(c1, c2)
    # handle all-NaN
    if np.all(np.isnan(dist)):
        print("Warning: all distances are NaN; defaulting max_dist=1")
        max_dist = 1.0
    else:
        max_dist = np.nanmax(dist)
    # fill features
    for i in range(N_seek):
        for j in range(N_job):
            # skills
            common = seeker_skills[i] & job_skills[j]
            missing = job_skills[j] - seeker_skills[i]
            penalty = 1 - len(missing)/len(job_skills[j])
            ms = sum(skill_idf.get(s,0) for s in common)
            ts = sum(skill_idf.get(s,0) for s in job_skills[j])
            F_raw[i,j,0] = penalty*(ms/ts if ts>0 else 0)
            # experience
            F_raw[i,j,1] = min(seekers_df.at[i,'years_experience']/max(jobs_df.at[j,'min_experience'],1),1.5)
            # salary
            diff = abs(jobs_df.at[j,'salary_offer'] - seekers_df.at[i,'salary'])
            F_raw[i,j,2] = 1 - np.log1p(diff)/np.log1p(max_sal-min_sal)
            # education
            F_raw[i,j,3] = seekers_df.at[i,'edu_value']/max_edu
            # sector
            F_raw[i,j,4] = (seek_sector_codes[i] == job_sector_codes[j])
            # contract
            F_raw[i,j,5] = (seek_cont_codes[i] == job_cont_codes[j])
            # edu history
            F_raw[i,j,6] = seekers_df.at[i,'edu_value']/20
            # location
            d = dist[i,j] if not np.isnan(dist[i,j]) else max_dist
            F_raw[i,j,7] = 1 - (d / (max_dist + 1e-8))
    # normalization
    mins = F_raw.min(axis=(0,1))
    maxs = F_raw.max(axis=(0,1))
    F = (F_raw - mins) / (maxs - mins + 1e-8)
    return F
# ---------- 4. Genetic Algorithm & Main ----------
def initialize_population(pop_size: int, N_seek: int, valid_jobs: np.ndarray) -> np.ndarray:
    pop = np.empty((pop_size, N_seek), dtype=int)
    for i in range(N_seek):
        choices = np.append(np.where(valid_jobs[i])[0], -1)
        pop[:,i] = np.random.choice(choices, pop_size)
    return pop

def calculate_fitness(pop: np.ndarray, F: np.ndarray, weights: np.ndarray) -> np.ndarray:
    scores = np.zeros(pop.shape[0])
    for k in range(pop.shape[0]):
        for i in range(pop.shape[1]):
            j = pop[k,i]
            if j>=0:
                scores[k] += F[i,j].dot(weights)
        scores[k] -= 0.1*np.sum(pop[k]==-1)
    return scores

def tournament_selection(pop: np.ndarray, fitness: np.ndarray, tournament_size: int = 3) -> np.ndarray:
    selected = np.empty_like(pop)
    for i in range(pop.shape[0]):
        contenders = np.random.choice(len(fitness), tournament_size, replace=False)
        winner = pop[contenders[np.argmax(fitness[contenders])]]
        selected[i] = winner
    return selected

def crossover(parent1: np.ndarray, parent2: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    if len(parent1)<=1:
        return parent1.copy(), parent2.copy()
    cut = np.random.randint(1,len(parent1))
    return (np.concatenate([parent1[:cut],parent2[cut:]]),
            np.concatenate([parent2[:cut],parent1[cut:]]))

def mutate(chromosome: np.ndarray, mutation_rate: float, valid_jobs: np.ndarray) -> np.ndarray:
    for i in range(len(chromosome)):
        if np.random.rand()<mutation_rate:
            choices = np.append(np.where(valid_jobs[i])[0],-1)
            chromosome[i] = np.random.choice(choices)
    return chromosome

def run_genetic_algorithm(F: np.ndarray, weights: np.ndarray, valid_jobs: np.ndarray,
                         pop_size: int = 50, generations: int = 100,
                         mutation_rate: float = 0.1, elite_size: int = 2) -> Tuple[np.ndarray, float]:
    N_seek = F.shape[0]
    pop = initialize_population(pop_size, N_seek, valid_jobs)
    best_score, best_solution = -np.inf, None
    for gen in range(generations):
        fitness = calculate_fitness(pop, F, weights)
        idx = np.argmax(fitness)
        if fitness[idx]>best_score:
            best_score, best_solution = fitness[idx], pop[idx].copy()
        selected = tournament_selection(pop, fitness)
        new_pop = []
        for i in range(0,pop_size-elite_size,2):
            c1,c2 = crossover(selected[i],selected[i+1])
            new_pop.extend([c1,c2])
        elites = pop[np.argsort(fitness)[-elite_size:]]
        new_pop.extend(elites)
        for i in range(len(new_pop)):
            new_pop[i] = mutate(new_pop[i],mutation_rate,valid_jobs)
        pop = np.array(new_pop)[:pop_size]
        print(f"Generation {gen+1}/{generations} | Best Score: {best_score:.2f}", end='\r')
    print()
    return best_solution, best_score

if __name__ == '__main__':
    seekers_df = load_data()
    print("=== Job Entry System ===")
    jobs_df = input_job_details(seekers_df)
    args = preprocess_data(seekers_df, jobs_df)
    F = calculate_features(*args)
    WEIGHTS = np.array([0.40,0.15,0.15,0.10,0.10,0.05,0.05,0.10], dtype=np.float32)
    valid = (
        (seekers_df['edu_rank'].values[:,None] >= jobs_df['edu_rank'].values[None,:]) &
        (seekers_df['years_experience'].values[:,None] >= jobs_df['min_experience'].values[None,:]) &
        np.array([[len(args[2][i]&args[3][j]) >= max(1,len(args[3][j])//2)
                   for j in range(len(jobs_df))] for i in range(len(seekers_df))])
    )
    best_sol, best_score = run_genetic_algorithm(F, WEIGHTS, valid)
    print("\n=== Matching Results ===")
    print(f"Best Overall Score: {best_score:.2f}\n")
    print("\n=== Top 5 Candidates per Job ===")
    for j in range(len(jobs_df)):
        scores = F[:,j].dot(WEIGHTS)
        valid_idx = np.where(valid[:,j])[0]
        ranked = sorted([(i, scores[i]) for i in valid_idx], key=lambda x: -x[1])[:5]
        print(f"\nJob {jobs_df.at[j,'job_id']}: ")
        for rank,(i,sc) in enumerate(ranked,1):
            print(f" {rank}. {seekers_df.at[i,'employee_Id']} (Score: {sc:.2%})")


=== Job Entry System ===

=== Enter Single Job Details ===
Press Enter for defaults or provide custom values

Generation 100/100 | Best Score: -747.71

=== Matching Results ===
Best Overall Score: -747.71


=== Top 5 Candidates per Job ===

Job 1: 
 1. 5594 (Score: 87.48%)
 2. 1943 (Score: 86.95%)
 3. 1246 (Score: 86.76%)
 4. 1094 (Score: 86.51%)
 5. 5882 (Score: 85.74%)
